# E4 — Fine-tuning VGG16 (Colab)

Reprise du **meilleur checkpoint** `E4_VGG16_best.keras`. Essai 1 : bloc 5 dégelé, LR `1e-5`, 5 epochs max. Même split et même preprocessing que le notebook initial. **Pas de test**.

Les CSV, le dataset décompressé et le checkpoint sont à la racine de `DS/projet`. Les résultats sont enregistrés dans `DS/projet/E4_finetuning/`.

Compatibilité Keras : copie locale du checkpoint avec correction des paramètres d’initialiseur non reconnus ; le fichier original Drive reste inchangé. Patience 2.

**Images :** lecture depuis `/content/`. La copie Drive → local est présente mais commentée par défaut. À réactiver seulement après réinitialisation du runtime.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import shutil
import time
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.vgg16 import preprocess_input
from tensorflow.keras.models import load_model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, CSVLogger
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

tf.keras.utils.set_random_seed(42)
PROJECT_DIR = Path('/content/drive/MyDrive/Apprentissage/DS/projet')
DRIVE_DATASET_ROOT = PROJECT_DIR / 'COVID-19_Radiography_Dataset'
DATASET_ROOT = Path('/content/COVID-19_Radiography_Dataset')
OUTPUT_DIR = PROJECT_DIR / 'E4_finetuning'
OUTPUT_DIR.mkdir(exist_ok=True)
CHECKPOINT = PROJECT_DIR / 'E4_VGG16_best.keras'
print('GPU :', tf.config.list_physical_devices('GPU'))
# COPIE FACULTATIVE : décommenter les lignes ci-dessous uniquement si
# le dossier local n'existe plus (nouveau runtime Colab).
# if not DATASET_ROOT.exists():
#     start = time.perf_counter()
#     shutil.copytree(DRIVE_DATASET_ROOT, DATASET_ROOT)
#     print(f"Copie locale en {(time.perf_counter()-start)/60:.1f} min")

assert DATASET_ROOT.is_dir(), (
    f"Images locales absentes : {DATASET_ROOT}. "
    "Décommenter la cellule de copie ci-dessus si le runtime a été réinitialisé."
)
print('Images utilisées :', DATASET_ROOT)


In [ ]:
train_df = pd.read_csv(PROJECT_DIR / 'train.csv')
val_df = pd.read_csv(PROJECT_DIR / 'validation.csv')

def local_filepath(original):
    normalized = str(original).replace('\\', '/')
    marker = 'COVID-19_Radiography_Dataset/'
    if marker not in normalized:
        raise ValueError(f'Chemin CSV inattendu : {original}')
    return str(DATASET_ROOT / normalized.split(marker, 1)[1])

train_df['filepath'] = train_df['filepath'].map(local_filepath)
val_df['filepath'] = val_df['filepath'].map(local_filepath)
print('Train :', len(train_df), 'Validation :', len(val_df))
assert Path(train_df['filepath'].iloc[0]).is_file(), train_df['filepath'].iloc[0]
assert Path(val_df['filepath'].iloc[0]).is_file(), val_df['filepath'].iloc[0]


In [ ]:
train_df['target'] = train_df['target'].astype(str)
val_df['target'] = val_df['target'].astype(str)
datagen = ImageDataGenerator(preprocessing_function=preprocess_input)
train_generator = datagen.flow_from_dataframe(train_df, x_col='filepath', y_col='target',
    target_size=(224,224), color_mode='rgb', class_mode='binary', batch_size=32, shuffle=True, seed=42)
val_generator = datagen.flow_from_dataframe(val_df, x_col='filepath', y_col='target',
    target_size=(224,224), color_mode='rgb', class_mode='binary', batch_size=32, shuffle=False)
print('Classes :', val_generator.class_indices)

In [ ]:
# Compatibilité Keras du checkpoint initial (copie locale, poids inchangés)
import json
import zipfile

COMPAT_CHECKPOINT = Path('/content/E4_VGG16_best_compatible.keras')

def nettoyer_config(obj):
    if isinstance(obj, dict):
        if obj.get('class_name') == 'GlorotUniform':
            obj.get('config', {}).pop('input_axes', None)
            obj.get('config', {}).pop('output_axes', None)
        for value in obj.values():
            nettoyer_config(value)
    elif isinstance(obj, list):
        for value in obj:
            nettoyer_config(value)

with zipfile.ZipFile(CHECKPOINT, 'r') as source:
    config = json.loads(source.read('config.json'))
    nettoyer_config(config)
    with zipfile.ZipFile(COMPAT_CHECKPOINT, 'w') as target:
        for member in source.infolist():
            data = (json.dumps(config).encode('utf-8') if member.filename == 'config.json'
                    else source.read(member.filename))
            target.writestr(member, data)

print('Copie compatible créée :', COMPAT_CHECKPOINT)


In [ ]:
# Configuration de l'essai : repartir du checkpoint initial à chaque nouvel essai.
EXPERIMENT_NAME = 'E4_FT_B5_LR1e-5'
LEARNING_RATE = 1e-5
EPOCHS = 5

model = load_model(COMPAT_CHECKPOINT, compile=False)
base_model = model.layers[0]  # VGG16 dans Sequential
base_model.trainable = True
for layer in base_model.layers:
    layer.trainable = layer.name.startswith('block5_')
print('Couches VGG16 entraînables :', [l.name for l in base_model.layers if l.trainable])

model.compile(optimizer=Adam(learning_rate=LEARNING_RATE),
              loss='binary_crossentropy', metrics=['accuracy'])
BEST_PATH = OUTPUT_DIR / f'{EXPERIMENT_NAME}_best.keras'
callbacks = [
    EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True),
    ModelCheckpoint(str(BEST_PATH), monitor='val_loss', save_best_only=True),
    CSVLogger(str(OUTPUT_DIR / f'{EXPERIMENT_NAME}_history.csv')),
]

In [ ]:
history = model.fit(train_generator, validation_data=val_generator,
                    epochs=EPOCHS, callbacks=callbacks)
print('Meilleur checkpoint :', BEST_PATH)

In [ ]:
best_model = load_model(BEST_PATH, compile=False)
val_generator.reset()
y_prob = best_model.predict(val_generator, verbose=1).ravel()
y_true = val_generator.classes
y_pred = (y_prob >= 0.5).astype(int)
labels = ['Non-COVID', 'COVID']
report = classification_report(y_true, y_pred, target_names=labels, output_dict=True, zero_division=0)
pd.DataFrame(report).T.to_csv(OUTPUT_DIR / f'{EXPERIMENT_NAME}_classification_report.csv')
pd.DataFrame({'y_true': y_true, 'y_proba': y_prob, 'y_pred': y_pred}).to_csv(OUTPUT_DIR / f'{EXPERIMENT_NAME}_predictions.csv', index=False)
print(classification_report(y_true, y_pred, target_names=labels, digits=4, zero_division=0))
cm = confusion_matrix(y_true, y_pred)
ConfusionMatrixDisplay(cm, display_labels=labels).plot(cmap='Blues')
plt.title(EXPERIMENT_NAME + ' — validation')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / f'{EXPERIMENT_NAME}_confusion_matrix.png', dpi=160, bbox_inches='tight')
plt.show()
print('Recall COVID :', report['COVID']['recall'], 'F1 COVID :', report['COVID']['f1-score'])

In [ ]:
hist = pd.read_csv(OUTPUT_DIR / f'{EXPERIMENT_NAME}_history.csv')
for metric in ('loss', 'accuracy'):
    plt.figure(figsize=(7, 4))
    plt.plot(hist[metric], label='Train')
    plt.plot(hist['val_' + metric], label='Validation')
    plt.title(f'{EXPERIMENT_NAME} — {metric}')
    plt.xlabel('Epoch')
    plt.legend()
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f'{EXPERIMENT_NAME}_{metric}.png', dpi=160)
    plt.show()